Imports:

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.patches import Patch
from matplotlib.ticker import ScalarFormatter, LogLocator
import country_converter as coco
import os
import plotly.express as px
from scipy import stats

import requests
import io
import zipfile

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm.auto import tqdm
tqdm.pandas()

Importing csv's

In [2]:
weather_df  = pd.read_csv('weather_london.csv')
bikes_df     = pd.read_csv('rentals2022.csv')
metro_df    = pd.read_csv('metrogebruik_londen_2022.csv')

C:\Users\jurre\AppData\Local\Temp\ipykernel_25228\499050194.py:2: DtypeWarning: Columns (0: bike_model) have mixed types. Specify dtype option on import or set low_memory=False.
  bikes_df     = pd.read_csv('rentals2022.csv')


converting station names to lat/lon data:

In [3]:
geolocator = Nominatim(user_agent="my_london_map_2026", timeout=10)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1, max_retries=1, error_wait_seconds=5)

unique = bikes_df["start_station"].drop_duplicates()
results = {}
for addr in tqdm(unique):
    loc = geocode(addr + ", London, UK")
    results[addr] = (loc.latitude, loc.longitude) if loc else (None, None)



  0%|          | 0/814 [00:00<?, ?it/s]

In [4]:
import json
with open("station_coords.json", "w") as f:
    json.dump(results, f)

coords = pd.DataFrame.from_dict(results, orient="index", columns=["lat", "lon"])

bikes_df["lat"] = bikes_df["start_station"].map(coords["lat"])
bikes_df["lon"] = bikes_df["start_station"].map(coords["lon"])

# Check which stations failed (unique stations, not millions of rows)
print(coords[coords["lat"].isna()])


                                   lat  lon
Courland Grove, Wandsworth Road    NaN  NaN
Taviton Street, Bloomsbury         NaN  NaN
Ampton Street , Clerkenwell        NaN  NaN
Denyer Street, Knightsbridge       NaN  NaN
Wellington Arch, Hyde Park         NaN  NaN
...                                ...  ...
Lansdown Drive, Hackney Central    NaN  NaN
Lansdowne Drive, Hackney Central   NaN  NaN
Portugal Street, Holborn           NaN  NaN
Gower Place , Euston               NaN  NaN
Coomer Place, West Kensington_OLD  NaN  NaN

[183 rows x 2 columns]


In [5]:
import re
import difflib

# 1. Download all docking stations from TfL (no API key needed for light use)
resp = requests.get("https://api.tfl.gov.uk/BikePoint", timeout=60)
resp.raise_for_status()

stations = pd.DataFrame(
    [{"name": s["commonName"], "lat_api": s["lat"], "lon_api": s["lon"]} for s in resp.json()]
)

stations = pd.DataFrame(
    [{"name": s["commonName"], "lat_api": s["lat"], "lon_api": s["lon"]} for s in resp.json()]
)

# 2. Normalise names so odd spacing ("Ampton Street , Clerkenwell") still matches
def norm(s):
    s = re.sub(r"\s*,\s*", ", ", str(s))   # one space after each comma, none before
    s = re.sub(r"\s+", " ", s)             # collapse repeated spaces
    return s.strip().lower()

stations["key"] = stations["name"].map(norm)
stations = stations.drop_duplicates("key").set_index("key")

bikes_df["key"] = bikes_df["start_station"].map(norm)

# 3. Exact match on the normalised name, only for rows still missing coordinates
missing = bikes_df["lat"].isna() | bikes_df["lon"].isna()
bikes_df.loc[missing, "lat"] = bikes_df.loc[missing, "key"].map(stations["lat_api"])
bikes_df.loc[missing, "lon"] = bikes_df.loc[missing, "key"].map(stations["lon_api"])

# 4. Fuzzy match for whatever is still missing, once per unique name
still = bikes_df["lat"].isna() | bikes_df["lon"].isna()
missing_keys = bikes_df.loc[still, "key"].unique()
keys = list(stations.index)

fuzzy = {}
for k in missing_keys:
    match = difflib.get_close_matches(k, keys, n=1, cutoff=0.85)
    if match:
        fuzzy[k] = (stations.at[match[0], "lat_api"], stations.at[match[0], "lon_api"])
        print(f"fuzzy: {k!r} -> {stations.at[match[0], 'name']!r}")

fuzzy_df = pd.DataFrame.from_dict(fuzzy, orient="index", columns=["lat", "lon"])

bikes_df.loc[still, "lat"] = bikes_df.loc[still, "key"].map(fuzzy_df["lat"])
bikes_df.loc[still, "lon"] = bikes_df.loc[still, "key"].map(fuzzy_df["lon"])

# 5. Check the result
print("Still missing:", bikes_df["lat"].isna().sum())
print(bikes_df[bikes_df["lat"].isna()][["start_station"]])

bikes_df = bikes_df.drop(columns="key")


fuzzy: 'lansdown drive, hackney central' -> 'Lansdowne Drive, Hackney Central'
fuzzy: 'coomer place, west kensington_old' -> 'Coomer Place, West Kensington'
Still missing: 102278
                            start_station
70          Harriet Street, Knightsbridge
474        Little Argyll Street, West End
475        Little Argyll Street, West End
3125      Earnshaw Street , Covent Garden
3126      Earnshaw Street , Covent Garden
...                                   ...
11419093    St. Martin's Street, West End
11419107    St. Martin's Street, West End
11419123    Harriet Street, Knightsbridge
11419140       King Edward Walk, Waterloo
11419241  Earnshaw Street , Covent Garden

[102278 rows x 1 columns]


In [6]:
print(bikes_df["start_station_id"].nunique())

missing_stations = bikes_df[bikes_df["lat"].isna()]

print(missing_stations["start_station_id"].nunique())           # 12 out of 1603 missing is not so bad, can drop those. 

1603
12


In [7]:
bikes2_df = bikes_df.dropna(subset=["lat", "lon"])

In [8]:
from sklearn.neighbors import BallTree

# 1. Unique bike stations only
bikes = (
    bikes2_df[["start_station", "lat", "lon"]]
    .dropna(subset=["lat", "lon"])
    .drop_duplicates("start_station")
    .rename(columns={"start_station": "name"})
    .reset_index(drop=True)
)

metro = (
    metro_df[["station", "lat", "lon"]]
    .dropna(subset=["lat", "lon"])
    .rename(columns={"station": "name"})
    .reset_index(drop=True)
)

# 2. Distance to the nearest metro station (haversine, in meters)
EARTH_RADIUS_M = 6_371_000
tree = BallTree(np.radians(metro[["lat", "lon"]].values), metric="haversine")
dist, idx = tree.query(np.radians(bikes[["lat", "lon"]].values), k=1)

bikes["dist_to_metro_m"] = dist[:, 0] * EARTH_RADIUS_M
bikes["nearest_metro"] = metro["name"].values[idx[:, 0]]

# 3. Categories: <=200 close, 200-500 medium, >500 far
bikes["metro_distance"] = pd.cut(
    bikes["dist_to_metro_m"],
    bins=[0, 200, 500, np.inf],
    labels=["Close (≤200 m)", "Medium (200-500 m)", "Far (>500 m)"],
    include_lowest=True,
)

print(bikes["metro_distance"].value_counts())

# 4. Combine with metro stations for the map
metro["metro_distance"] = "Metro station"
combined = pd.concat([metro, bikes], ignore_index=True)

color_map = {
    "Metro station": "#1f1f1f",
    "Close (≤200 m)": "#2ca02c",
    "Medium (200-500 m)": "#ff7f0e",
    "Far (>500 m)": "#d62728",
}

fig = px.scatter_mapbox(
    combined, lat="lat", lon="lon",
    color="metro_distance",
    color_discrete_map=color_map,
    category_orders={"metro_distance": list(color_map)},
    hover_name="name",
    hover_data={"dist_to_metro_m": ":.0f", "nearest_metro": True,
                "lat": False, "lon": False, "metro_distance": False},
    zoom=10, height=600, mapbox_style="open-street-map",
)
fig.update_traces(marker=dict(size=8))
fig.update_layout(legend_title_text="")
fig.show()

metro_distance
Medium (200-500 m)    405
Far (>500 m)          236
Close (≤200 m)        167
Name: count, dtype: int64


C:\Users\jurre\AppData\Local\Temp\ipykernel_25228\3361559343.py:48: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


Visuals for testing hypothesis

In [9]:
# 1. Rentals per station (one pass over the 10M rows)
rentals = (
    bikes2_df.groupby("start_station")
    .agg(rentals=("start_station", "size"),
         first_rental=("start_date", "min"),
         last_rental=("start_date", "max"))
)

# Normalise for stations that were open for different lengths of time
rentals["active_days"] = (
    (pd.to_datetime(rentals["last_rental"]) - pd.to_datetime(rentals["first_rental"]))
    .dt.days.clip(lower=1)
)
rentals["rentals_per_day"] = rentals["rentals"] / rentals["active_days"]

# 2. Join onto the station table
st = bikes.merge(rentals, left_on="name", right_index=True, how="inner")
order = ["Close (≤200 m)", "Medium (200-500 m)", "Far (>500 m)"]
st["metro_distance"] = pd.Categorical(st["metro_distance"], categories=order, ordered=True)

# 2.1 added after concluding that there is an outlier in the data

MAX_DIST_M = 10_000

# 2.2 Look at what you're dropping before you drop it
far_away = bikes[bikes["dist_to_metro_m"] > MAX_DIST_M]
print(f"{len(far_away)} stations farther than 10 km:")
print(far_away[["name", "lat", "lon", "dist_to_metro_m", "nearest_metro"]]
      .sort_values("dist_to_metro_m", ascending=False))

# 2.3 Station-level tables (used for the map and the correlation analysis)
bikes = bikes[bikes["dist_to_metro_m"] <= MAX_DIST_M].copy()
st = st[st["dist_to_metro_m"] <= MAX_DIST_M].copy()

# 2.3. Optional: also drop their rides from the full dataframe
bikes2_df["dist_to_metro_m"] = bikes2_df["start_station"].map(
    bikes.set_index("name")["dist_to_metro_m"]
)
before = len(bikes2_df)
bikes2_df = bikes2_df[bikes2_df["dist_to_metro_m"].notna()].copy()
print(f"Dropped {before - len(bikes2_df):,} rides")

# 3. Summary per category
summary = st.groupby("metro_distance", observed=True)["rentals_per_day"].agg(
    stations="count", mean="mean", median="median", std="std"
).round(1)
print(summary)

# 4. Statistical tests
# Continuous distance vs rentals (rank-based, robust to skew and outliers)
rho, p_rho = stats.spearmanr(st["dist_to_metro_m"], st["rentals_per_day"])
print(f"Spearman rho = {rho:.3f}, p = {p_rho:.4f}")

# Do the three categories differ?
groups = [g["rentals_per_day"].values for _, g in st.groupby("metro_distance", observed=True)]
h, p_kw = stats.kruskal(*groups)
print(f"Kruskal-Wallis H = {h:.2f}, p = {p_kw:.4f}")

# 5. Visuals
# a) Box plot per category
fig1 = px.box(
    st, x="metro_distance", y="rentals_per_day", color="metro_distance",
    category_orders={"metro_distance": order}, points="all", hover_name="name",
    log_y=True, height=500,
    labels={"rentals_per_day": "Rentals per day (log scale)", "metro_distance": ""},
    title="Daily rentals per station by distance to nearest metro",
)
fig1.update_layout(showlegend=False)
fig1.show()

# b) Scatter of exact distance vs rentals, with trend line
fig2 = px.scatter(
    st, x="dist_to_metro_m", y="rentals_per_day", color="metro_distance",
    category_orders={"metro_distance": order}, hover_name="name",
    trendline="lowess", trendline_scope="overall", trendline_color_override="black",
    log_y=True, height=500,
    labels={"dist_to_metro_m": "Distance to nearest metro (m)",
            "rentals_per_day": "Rentals per day (log scale)"},
    title="Distance to metro vs. rentals per station",
)
fig2.show()

# c) Average with the spread made visible
fig3 = px.bar(
    summary.reset_index(), x="metro_distance", y="median",
    error_y="std", text="stations", height=450,
    labels={"median": "Median rentals per day", "metro_distance": ""},
    title="Median rentals per day (label = number of stations)",
)
fig3.show()

1 stations farther than 10 km:
                     name        lat       lon  dist_to_metro_m nearest_metro
97  Park Lane , Hyde Park  53.800021 -1.558196    241466.210924       Chesham
Dropped 37,674 rides
                    stations  mean  median   std
metro_distance                                  
Close (≤200 m)           167  51.0    43.7  31.2
Medium (200-500 m)       405  36.8    32.9  21.0
Far (>500 m)             235  34.2    32.1  18.2
Spearman rho = -0.224, p = 0.0000
Kruskal-Wallis H = 44.48, p = 0.0000


In [23]:
bikes2_df.to_csv("bikes_geocoded.csv", index=False)

In [25]:
bikes2_df.to_parquet("stations.parquet", index=False)   # save

In [24]:
print(bikes2_df.info())

<class 'pandas.DataFrame'>
Index: 11279342 entries, 0 to 11419293
Data columns (total 14 columns):
 #   Column            Dtype  
---  ------            -----  
 0   rental_id         int64  
 1   duration_s        float64
 2   bike_id           int64  
 3   end_date          str    
 4   end_station_id    float64
 5   end_station       str    
 6   start_date        str    
 7   start_station_id  float64
 8   start_station     str    
 9   source_file       str    
 10  bike_model        str    
 11  lat               float64
 12  lon               float64
 13  dist_to_metro_m   float64
dtypes: float64(6), int64(2), str(6)
memory usage: 2.7 GB
None
